# Trustee Demo: Detecting Shortcut Learning

## Introduction

In this demo, you'll learn how to use **Trustee** to explain machine learning models and detect **shortcut learning**.

### What is Shortcut Learning?

**Shortcut learning** occurs when a model relies on spurious correlations instead of true patterns. For example:
- A skin cancer classifier that uses rulers in images instead of skin patterns
- An intrusion detection system that uses packet arrival time instead of content

### Demo: Moon & Stars

- **Task:** Classify images as "moon" (circle) or "star" (star shape)
- **Shortcut:** Training images have spatially biased positions (moons on left, stars on right)
- **Goal:** Use Trustee to detect if the model learned position (shortcut) vs. shape (true pattern)

## Setup: Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
from trustee import ClassificationTrustee
from sklearn import tree
import cv2

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✓ Libraries imported successfully!")

## Step 1: Create Synthetic Dataset with Spatial Bias

In [ ]:
def create_synthetic_shapes():
    """Create simple moon (circle) and star shapes"""
    # Create moon (circle)
    moon = np.zeros((50, 50), dtype=np.uint8)
    cv2.circle(moon, (25, 25), 20, 255, -1)
    
    # Create star (simple 5-pointed star)
    star = np.zeros((50, 50), dtype=np.uint8)
    pts = np.array([[25, 5], [30, 20], [45, 20], [33, 30], 
                     [38, 45], [25, 35], [12, 45], [17, 30], [5, 20], [20, 20]], np.int32)
    cv2.fillPoly(star, [pts], 255)
    
    return moon, star

IMG_SIZE = 50
moon_img, star_img = create_synthetic_shapes()

# Visualize the base shapes
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(moon_img, cmap='gray')
axes[0].set_title('Moon (Class 0)', fontsize=12)
axes[0].axis('off')
axes[1].imshow(star_img, cmap='gray')
axes[1].set_title('Star (Class 1)', fontsize=12)
axes[1].axis('off')
plt.tight_layout()
plt.show()

print("✓ Base shapes created")

In [ ]:
def create_biased_dataset(moon_img, star_img, n_samples=500, bias_strength=0.7):
    """
    Create dataset where position is correlated with class (shortcut learning)
    
    Args:
        moon_img: Base moon image
        star_img: Base star image
        n_samples: Number of samples per class
        bias_strength: How much spatial bias (0=none, 1=complete)
    """
    canvas_size = 100
    X = []
    y = []
    
    for i in range(n_samples * 2):
        canvas = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
        
        # Alternate between moon and star
        if i < n_samples:
            img = moon_img
            label = 0
            # Bias: moons appear more on the left
            if np.random.rand() < bias_strength:
                x_pos = np.random.randint(0, max(1, (canvas_size - IMG_SIZE) // 3))
            else:
                x_pos = np.random.randint(0, canvas_size - IMG_SIZE)
        else:
            img = star_img
            label = 1
            # Bias: stars appear more on the right
            if np.random.rand() < bias_strength:
                min_pos = max((canvas_size - IMG_SIZE) * 2 // 3, 1)
                max_pos = canvas_size - IMG_SIZE
                if min_pos < max_pos:
                    x_pos = np.random.randint(min_pos, max_pos)
                else:
                    x_pos = max_pos - 1
            else:
                x_pos = np.random.randint(0, canvas_size - IMG_SIZE)
        
        y_pos = np.random.randint(0, canvas_size - IMG_SIZE)
        
        # Place image on canvas
        canvas[y_pos:y_pos+IMG_SIZE, x_pos:x_pos+IMG_SIZE] = img
        
        X.append(canvas.flatten())
        y.append(label)
    
    return np.array(X, dtype=np.float32) / 255.0, np.array(y)

def create_unbiased_dataset(moon_img, star_img, n_samples=500):
    """Create dataset with no positional bias (for testing)"""
    return create_biased_dataset(moon_img, star_img, n_samples, bias_strength=0.0)

# Create biased training set and unbiased test set
print("Creating biased training dataset...")
X_train_biased, y_train_biased = create_biased_dataset(moon_img, star_img, n_samples=500, bias_strength=0.8)

print("Creating unbiased test dataset...")
X_test_unbiased, y_test_unbiased = create_unbiased_dataset(moon_img, star_img, n_samples=200)

print(f"✓ Training set: {X_train_biased.shape}, Test set: {X_test_unbiased.shape}")

In [ ]:
# Visualize some examples from biased dataset
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    axes[0, i].imshow(X_train_biased[i].reshape(100, 100), cmap='gray')
    axes[0, i].set_title(f'Moon (biased left)', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(X_train_biased[500 + i].reshape(100, 100), cmap='gray')
    axes[1, i].set_title(f'Star (biased right)', fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('Biased Training Examples (Notice spatial patterns!)', fontsize=14)
plt.tight_layout()
plt.show()

## Step 2: Train a Neural Network

In [ ]:
class SimpleNN(nn.Module):
    """Simple fully-connected neural network"""
    def __init__(self, input_size=10000, hidden_size=256):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size, 2)
    
    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x
    
    def predict(self, X):
        """Sklearn-compatible predict method for Trustee"""
        self.eval()
        with torch.no_grad():
            if hasattr(X, 'values'):
                X = X.values
            if not isinstance(X, np.ndarray):
                X = np.array(X)
            
            X_tensor = torch.FloatTensor(X)
            outputs = self.forward(X_tensor)
            _, predicted = torch.max(outputs.data, 1)
            return predicted.numpy()

# Initialize model
model = SimpleNN(input_size=100*100, hidden_size=256)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("✓ Model initialized")

In [ ]:
# Train the model
print("Training neural network...")
num_epochs = 50
batch_size = 32

X_train_tensor = torch.FloatTensor(X_train_biased)
y_train_tensor = torch.LongTensor(y_train_biased)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for i in range(0, len(X_train_tensor), batch_size):
        batch_X = X_train_tensor[i:i+batch_size]
        batch_y = y_train_tensor[i:i+batch_size]
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(X_train_tensor):.4f}')

# Evaluate
train_pred = model.predict(X_train_biased)
test_pred = model.predict(X_test_unbiased)

print(f"\n{'='*60}")
print(f"Training Accuracy (biased data): {accuracy_score(y_train_biased, train_pred):.3f}")
print(f"Test Accuracy (unbiased data): {accuracy_score(y_test_unbiased, test_pred):.3f}")
print(f"{'='*60}")
print("\n⚠️  If test accuracy is much lower than training, the model likely learned the spatial shortcut!")

## Step 3: Use Trustee to Explain the Model

Now we'll use Trustee to extract a decision tree that explains the neural network's behavior.

In [ ]:
# Initialize Trustee with the neural network
print("Extracting decision tree explanation with Trustee...")
trustee = ClassificationTrustee(expert=model)

# Fit Trustee on the training data
trustee.fit(
    X_train_biased,
    y_train_biased,
    num_iter=10,
    num_stability_iter=5,
    samples_size=0.3,
    verbose=True
)

# Extract explanation
dt, pruned_dt, agreement, reward = trustee.explain()

print(f"\n{'='*60}")
print(f"Trustee Explanation Metrics:")
print(f"{'='*60}")
print(f"Agreement (training fidelity): {agreement:.3f}")
print(f"Reward (validation fidelity): {reward:.3f}")
print(f"Decision Tree nodes: {dt.tree_.node_count}")
print(f"Pruned Decision Tree nodes: {pruned_dt.tree_.node_count}")

In [ ]:
# Visualize the pruned decision tree
fig, ax = plt.subplots(figsize=(20, 10))
tree.plot_tree(
    pruned_dt,
    class_names=['Moon', 'Star'],
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax
)
plt.title("Trustee Decision Tree Explanation (Pruned)", fontsize=16, pad=20)
plt.tight_layout()
plt.show()

print(f"\nTree depth: {pruned_dt.get_depth()}")
print(f"Number of leaves: {pruned_dt.get_n_leaves()}")
print(f"Total nodes: {pruned_dt.tree_.node_count}")

## Step 4: Analyze - Did the Model Learn the Shortcut?

We'll analyze which pixels (features) the model relies on most.

In [ ]:
# Extract feature importance
feature_importance = pruned_dt.feature_importances_

# Reshape to understand spatial importance
importance_map = feature_importance.reshape(100, 100)

# Visualize which pixels the model relies on
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Heatmap of feature importance
im = axes[0].imshow(importance_map, cmap='hot', interpolation='nearest')
axes[0].set_title('Trustee Feature Importance Heatmap', fontsize=14)
axes[0].set_xlabel('X Position')
axes[0].set_ylabel('Y Position')
plt.colorbar(im, ax=axes[0])

# Highlight left vs right importance (detecting spatial bias)
left_importance = np.sum(importance_map[:, :33])  # Left third
right_importance = np.sum(importance_map[:, 66:])  # Right third
center_importance = np.sum(importance_map[:, 33:66])  # Center

spatial_importance = [left_importance, center_importance, right_importance]
axes[1].bar(['Left\n(Moon bias)', 'Center', 'Right\n(Star bias)'], spatial_importance, 
            color=['blue', 'gray', 'orange'])
axes[1].set_title('Spatial Feature Importance', fontsize=14)
axes[1].set_ylabel('Total Importance')

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print("Shortcut Learning Detection:")
print(f"{'='*60}")
if left_importance > center_importance or right_importance > center_importance:
    print("⚠️  WARNING: Model shows high importance on spatial position!")
    print("   This indicates SHORTCUT LEARNING - the model learned position instead of shape.")
else:
    print("✓ Model appears to focus on central features (shape).")
    
print(f"\nLeft region importance: {left_importance:.4f}")
print(f"Center region importance: {center_importance:.4f}")
print(f"Right region importance: {right_importance:.4f}")

## Key Takeaways

1. **Shortcut Learning is Real**: Models can achieve high training accuracy by learning spurious correlations
2. **Trustee Reveals Shortcuts**: The decision tree explanation shows which features the model actually uses
3. **Spatial Bias Detection**: Feature importance heatmap clearly shows if the model learned position vs. shape
4. **Test on Unbiased Data**: Performance drop on unbiased test set indicates shortcut learning

### Why This Matters

- **Security Models**: An IDS might learn packet timestamps instead of malicious patterns
- **Medical Diagnosis**: A model might learn hospital equipment markers instead of disease symptoms
- **Fairness**: Models might learn protected attributes as shortcuts

### Next Steps

Now you'll apply these techniques to a real-world problem in the exercise notebook!